# Эксперименты со сходимостью CV-моделей

Этот notebook нужен для сравнения того, как быстро CV-регрессор сходится при разных гиперпараметрах, архитектурах и функциях потерь. В качестве варианта данных используется `x_y_theta`: модель предсказывает `x`, `y` и угол через `sin(theta)`, `cos(theta)`. Для дальнейшей RL-интеграции результаты считаются под режим `hybrid`, где CV отвечает за позицию и угол, а остальные компоненты состояния могут приходить из окружения.

## Импорты и пути

Пути вычисляются от корня проекта, поэтому notebook можно запускать из VSCode независимо от текущей рабочей папки ядра.

In [1]:
import json
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "train_cv.py").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Не удалось найти корень проекта с train_cv.py и src/.")


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from train_cv import (  # noqa: E402
    LunarLanderCVDataset,
    _seed_everything,
    build_model,
    load_integration_config,
    make_loaders,
    run_epoch,
)
from lunar_lander_cvrl.models.cv import CV_MODEL_TYPES  # noqa: E402

print(f"Корень проекта: {PROJECT_ROOT}")
print(f"Доступные CV-модели: {CV_MODEL_TYPES}")

Корень проекта: C:\Users\Ilya\Desktop\LunarLander
Доступные CV-модели: ('resnet18', 'simple-cnn')


## Базовые настройки

`LIMIT_SAMPLES = 0` означает обучение на всём датасете. Для быстрой проверки пайплайна можно поставить небольшое значение, например `1000` или `5000`.

In [2]:
INTEGRATION = "x_y_theta"
METADATA_PATH = PROJECT_ROOT / "data" / "cv_integrations" / INTEGRATION / "metadata.json"

PREDICTION_MODE = "hybrid"
ANGLE_TARGET = "sincos"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LIMIT_SAMPLES = 0
NUM_WORKERS = 0
SAVE_BEST_CHECKPOINT = True

RESULTS_DIR = PROJECT_ROOT / "runs" / "experiments" / "cv_convergence"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints" / "cv" / "experiments"
WANDB_DIR = PROJECT_ROOT / "runs" / "wandb"

WANDB_ENABLED = True
WANDB_PROJECT = "lunar-lander-cvrl-cv"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
WANDB_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Metadata: {METADATA_PATH}")
print(f"Prediction mode для будущей RL-интеграции: {PREDICTION_MODE}")

Device: cuda
Metadata: C:\Users\Ilya\Desktop\LunarLander\data\cv_integrations\x_y_theta\metadata.json
Prediction mode для будущей RL-интеграции: hybrid


## Проверка датасета

Для режима `hybrid` с предсказанием угла используем `x_y_theta`. На выходе модели будет четыре значения: `x`, `y`, `sin_theta`, `cos_theta`.

In [3]:
config = load_integration_config(INTEGRATION, str(METADATA_PATH))
dataset = LunarLanderCVDataset(
    config,
    angle_target=ANGLE_TARGET,
    augment=True,
    particle_prob=0.35,
    seed=42,
)
if LIMIT_SAMPLES > 0:
    dataset.samples = dataset.samples[:LIMIT_SAMPLES]

assert config.target_columns == ["x", "y", "theta"]
assert dataset.output_columns == ["x", "y", "sin_theta", "cos_theta"]

print(f"Samples: {len(dataset)}")
print(f"Images: {config.images_dir}")
print(f"Labels: {config.labels_file}")
print(f"Target columns: {config.target_columns}")
print(f"Model outputs: {dataset.output_columns}")

labels_df = pd.read_csv(config.labels_file, usecols=["x", "y", "theta"])
if LIMIT_SAMPLES > 0:
    labels_df = labels_df.head(LIMIT_SAMPLES)


Samples: 15000
Images: C:\Users\Ilya\Desktop\LunarLander\data\images
Labels: C:\Users\Ilya\Desktop\LunarLander\data\labels.csv
Target columns: ['x', 'y', 'theta']
Model outputs: ['x', 'y', 'sin_theta', 'cos_theta']


## W&B offline-логирование

Если установлен пакет `wandb`, каждый запуск будет писаться в offline-режиме в `runs/wandb`. Внутри папки запуска появится локальный `.wandb` файл, который можно открыть расширением VSCode. Если `wandb` не установлен, эксперимент продолжит работать без логирования.

In [4]:
def get_wandb():
    if not WANDB_ENABLED:
        return None
    os.environ.setdefault("WANDB_MODE", "offline")
    os.environ.setdefault("WANDB_SILENT", "true")
    os.environ.setdefault("WANDB_DISABLE_CODE", "true")
    os.environ.setdefault("WANDB_DISABLE_GIT", "true")
    os.environ.setdefault("WANDB_DISABLE_SERVICE", "true")
    try:
        import wandb
    except ImportError:
        print("wandb не установлен: логирование будет пропущено. Установить можно командой: pip install wandb")
        return None
    return wandb


def make_wandb_settings():
    if wandb is None:
        return None
    base_kwargs = {
        "start_method": "thread",
        "init_timeout": 120,
        "silent": True,
        "disable_git": True,
        "disable_code": True,
    }
    for wait_key in ("x_service_wait", "_service_wait"):
        try:
            return wandb.Settings(**base_kwargs, **{wait_key: 120})
        except TypeError:
            continue
    return wandb.Settings(**base_kwargs)


def init_wandb_run(run_name: str, run_config: dict):
    if wandb is None:
        return None
    try:
        return wandb.init(
            project=WANDB_PROJECT,
            name=run_name,
            config=run_config,
            mode="offline",
            dir=str(WANDB_DIR),
            reinit=True,
            settings=make_wandb_settings(),
        )
    except Exception as exc:
        print(f"W&B offline не запустился ({type(exc).__name__}: {exc}). Продолжаю без .wandb-лога.")
        return None


def safe_wandb_log(run, row: dict, step: int) -> None:
    if run is None or wandb is None:
        return
    try:
        wandb.log(row, step=step)
    except Exception as exc:
        print(f"W&B log пропущен ({type(exc).__name__}: {exc}).")


def safe_wandb_finish(run) -> None:
    if run is None:
        return
    try:
        run.finish()
    except Exception as exc:
        print(f"W&B finish пропущен ({type(exc).__name__}: {exc}).")


def find_wandb_file(run) -> str | None:
    if run is None:
        return None
    run_dir = Path(run.dir).resolve().parent
    candidates = sorted(run_dir.glob("*.wandb"))
    return str(candidates[0]) if candidates else None


wandb = get_wandb()

## Функции потерь и метрики

Для честного сравнения функций потерь дополнительно считаем одинаковые validation-метрики: MSE и MAE по каждому выходу модели.

In [5]:
class WeightedMSELoss(nn.Module):
    def __init__(self, weights: list[float]):
        super().__init__()
        self.register_buffer("weights", torch.tensor(weights, dtype=torch.float32))

    def forward(self, preds: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        weights = self.weights.to(device=preds.device, dtype=preds.dtype)
        return ((preds - targets) ** 2 * weights).mean()


LOSS_FACTORIES = {
    "mse": nn.MSELoss,
    "mae": nn.L1Loss,
    "smooth_l1": lambda: nn.SmoothL1Loss(beta=0.05),
    "weighted_pose_mse": lambda: WeightedMSELoss([1.0, 1.0, 0.5, 0.5]),
}


def evaluate_component_metrics(model, loader, device, output_columns):
    model.eval()
    sq_error = torch.zeros(len(output_columns), dtype=torch.float64)
    abs_error = torch.zeros(len(output_columns), dtype=torch.float64)
    total = 0

    with torch.inference_mode():
        for images, targets in loader:
            images = images.to(device)
            targets = targets.to(device)
            preds = model(images)
            error = (preds - targets).detach().cpu().double()
            sq_error += (error ** 2).sum(dim=0)
            abs_error += error.abs().sum(dim=0)
            total += images.size(0)

    mse = sq_error / max(1, total)
    mae = abs_error / max(1, total)
    metrics = {
        "val_mse_total": float(mse.mean().item()),
        "val_mae_total": float(mae.mean().item()),
    }
    for idx, column in enumerate(output_columns):
        metrics[f"val_mse/{column}"] = float(mse[idx].item())
        metrics[f"val_mae/{column}"] = float(mae[idx].item())
    return metrics

## Один эксперимент

Функция ниже обучает одну конфигурацию, сохраняет историю в `runs/experiments/cv_convergence`, при необходимости сохраняет лучший checkpoint и пишет метрики в W&B offline.

In [6]:
def make_optimizer(name: str, parameters, lr: float, weight_decay: float):
    if name == "adam":
        return torch.optim.Adam(parameters, lr=lr, weight_decay=weight_decay)
    if name == "adamw":
        return torch.optim.AdamW(parameters, lr=lr, weight_decay=weight_decay)
    raise ValueError(f"Неизвестный optimizer: {name}")


def train_one_experiment(exp: dict) -> dict:
    run_name = exp["name"]
    seed = int(exp.get("seed", 42))
    _seed_everything(seed)

    run_dataset = LunarLanderCVDataset(
        config,
        angle_target=ANGLE_TARGET,
        augment=bool(exp.get("augment", True)),
        particle_prob=float(exp.get("particle_prob", 0.35)),
        seed=seed,
    )
    limit_samples = int(exp.get("limit_samples", LIMIT_SAMPLES))
    if limit_samples > 0:
        run_dataset.samples = run_dataset.samples[:limit_samples]

    train_loader, val_loader = make_loaders(
        dataset=run_dataset,
        val_ratio=float(exp.get("val_ratio", 0.2)),
        batch_size=int(exp.get("batch_size", 32)),
        num_workers=int(exp.get("num_workers", NUM_WORKERS)),
        seed=seed,
    )

    model = build_model(exp["model_type"], out_dim=len(run_dataset.output_columns)).to(DEVICE)
    criterion = LOSS_FACTORIES[exp["loss"]]()
    optimizer = make_optimizer(
        name=exp.get("optimizer", "adamw"),
        parameters=model.parameters(),
        lr=float(exp["lr"]),
        weight_decay=float(exp.get("weight_decay", 0.0)),
    )

    run_config = {
        **exp,
        "integration": INTEGRATION,
        "prediction_mode": PREDICTION_MODE,
        "angle_target": ANGLE_TARGET,
        "model_output_columns": run_dataset.output_columns,
        "samples": len(run_dataset),
        "device": str(DEVICE),
    }

    wb_run = init_wandb_run(run_name, run_config)

    run_dir = RESULTS_DIR / run_name
    ckpt_dir = CHECKPOINTS_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    best_val_mse = float("inf")
    history = []
    started_at = time.time()

    for epoch in range(1, int(exp["epochs"]) + 1):
        train_loss = run_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_metrics = evaluate_component_metrics(model, val_loader, DEVICE, run_dataset.output_columns)
        val_loss = val_metrics["val_mse_total"]
        elapsed_sec = time.time() - started_at

        row = {
            "run_name": run_name,
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "lr": float(optimizer.param_groups[0]["lr"]),
            "elapsed_sec": elapsed_sec,
            **val_metrics,
        }
        history.append(row)

        safe_wandb_log(wb_run, row, step=epoch)

        if SAVE_BEST_CHECKPOINT and val_loss < best_val_mse:
            best_val_mse = val_loss
            torch.save(model.state_dict(), ckpt_dir / "best_model.pth")

        print(
            f"{run_name} | epoch {epoch:02d} | "
            f"train_loss={train_loss:.6f} | val_mse={val_loss:.6f}"
        )

    history_df = pd.DataFrame(history)
    history_df.to_csv(run_dir / "history.csv", index=False)
    (run_dir / "config.json").write_text(json.dumps(run_config, indent=2, ensure_ascii=False), encoding="utf-8")

    wandb_file = find_wandb_file(wb_run)
    safe_wandb_finish(wb_run)

    summary = {
        "run_name": run_name,
        "best_val_mse": best_val_mse,
        "final_val_mse": float(history_df["val_mse_total"].iloc[-1]),
        "final_train_loss": float(history_df["train_loss"].iloc[-1]),
        "history_csv": str(run_dir / "history.csv"),
        "best_checkpoint": str(ckpt_dir / "best_model.pth") if SAVE_BEST_CHECKPOINT else None,
        "wandb_file": wandb_file,
    }
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
    return summary

## Сетка экспериментов

Начальная сетка небольшая: она сравнивает MSE, Smooth L1, взвешенный MSE и более простую CNN. Значения можно свободно расширять новыми learning rate, batch size, архитектурами и функциями потерь.

In [7]:
EXPERIMENTS = [
    {
        "name": "resnet18_mse_lr1e-5_bs32",
        "model_type": "resnet18",
        "loss": "mse",
        "optimizer": "adamw",
        "lr": 1e-5,
        "weight_decay": 0.0,
        "batch_size": 32,
        "epochs": 8,
        "seed": 42,
    },
    {
        "name": "resnet18_smoothl1_lr3e-5_bs32",
        "model_type": "resnet18",
        "loss": "smooth_l1",
        "optimizer": "adamw",
        "lr": 3e-5,
        "weight_decay": 1e-4,
        "batch_size": 32,
        "epochs": 8,
        "seed": 42,
    },
    {
        "name": "resnet18_weighted_pose_mse_lr1e-5_bs64",
        "model_type": "resnet18",
        "loss": "weighted_pose_mse",
        "optimizer": "adamw",
        "lr": 1e-5,
        "weight_decay": 1e-4,
        "batch_size": 64,
        "epochs": 8,
        "seed": 42,
    },
    {
        "name": "simplecnn_mse_lr1e-4_bs64",
        "model_type": "simple-cnn",
        "loss": "mse",
        "optimizer": "adamw",
        "lr": 1e-4,
        "weight_decay": 1e-4,
        "batch_size": 64,
        "epochs": 8,
        "seed": 42,
    },
]

pd.DataFrame(EXPERIMENTS)

,name,model_type,loss,optimizer,lr,weight_decay,batch_size,epochs,seed
0,resnet18_mse_lr1e-5_bs32,resnet18,mse,adamw,0.00001,0.0000,32,8,42
1,resnet18_smoothl1_lr3e-5_bs32,resnet18,smooth_l1,adamw,0.00003,0.0001,32,8,42
2,resnet18_weighted_pose_mse_lr1e-5_bs64,resnet18,weighted_pose_mse,adamw,0.00001,0.0001,64,8,42
3,simplecnn_mse_lr1e-4_bs64,simple-cnn,mse,adamw,0.00010,0.0001,64,8,42


## Запуск экспериментов

Ячейка ниже запускает всю сетку. Если нужно сначала проверить пайплайн, поставьте `LIMIT_SAMPLES` выше в небольшое значение или временно оставьте в `EXPERIMENTS` один запуск.

In [8]:
summaries = []
for experiment in EXPERIMENTS:
    summaries.append(train_one_experiment(experiment))

summary_df = pd.DataFrame(summaries).sort_values("best_val_mse")
summary_df.to_csv(RESULTS_DIR / "summary.csv", index=False)
summary_df

wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.


W&B offline не запустился (ServicePollForTokenError: Failed to read port info after 30.0 seconds.). Продолжаю без .wandb-лога.
resnet18_mse_lr1e-5_bs32 | epoch 01 | train_loss=0.280912 | val_mse=0.195963
resnet18_mse_lr1e-5_bs32 | epoch 02 | train_loss=0.105949 | val_mse=0.024941
resnet18_mse_lr1e-5_bs32 | epoch 03 | train_loss=0.028954 | val_mse=0.015407
resnet18_mse_lr1e-5_bs32 | epoch 04 | train_loss=0.022412 | val_mse=0.011529
resnet18_mse_lr1e-5_bs32 | epoch 05 | train_loss=0.019701 | val_mse=0.005914
resnet18_mse_lr1e-5_bs32 | epoch 06 | train_loss=0.017535 | val_mse=0.006824
resnet18_mse_lr1e-5_bs32 | epoch 07 | train_loss=0.016496 | val_mse=0.006417
resnet18_mse_lr1e-5_bs32 | epoch 08 | train_loss=0.014936 | val_mse=0.004461
W&B offline не запустился (ServicePollForTokenError: Failed to read port info after 30.0 seconds.). Продолжаю без .wandb-лога.
resnet18_smoothl1_lr3e-5_bs32 | epoch 01 | train_loss=0.260105 | val_mse=0.045370
resnet18_smoothl1_lr3e-5_bs32 | epoch 02 | train

KeyboardInterrupt: 

## График сходимости

График строится по сохранённым `history.csv`, поэтому его можно перезапускать отдельно после любых экспериментов.

In [ ]:
history_frames = []
for history_path in sorted(RESULTS_DIR.glob("*/history.csv")):
    history_frames.append(pd.read_csv(history_path))

if history_frames:
    all_history = pd.concat(history_frames, ignore_index=True)
    plt.figure(figsize=(10, 6))
    for run_name, run_history in all_history.groupby("run_name"):
        plt.plot(run_history["epoch"], run_history["val_mse_total"], marker="o", label=run_name)
    plt.xlabel("Epoch")
    plt.ylabel("Validation MSE")
    plt.yscale("log")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
else:
    print("Пока нет сохранённых history.csv. Сначала запустите эксперименты выше.")

## Пути к локальным W&B-файлам

После запуска экспериментов в таблице `summary_df` появится колонка `wandb_file`. Это путь к локальному `.wandb` файлу для просмотра в VSCode.

In [ ]:
if "summary_df" in globals():
    display(summary_df[["run_name", "best_val_mse", "wandb_file", "history_csv"]])
else:
    print("summary_df появится после запуска экспериментов.")